## Step 1: Check PDF Files in the Data Folder

In this step, we check the `data` folder and list all the files stored inside it.

The `os` module is used to interact with the operating system and access files and directories.

In [1]:
import os

pdf_folder = r"C:\Data Science Course\RAG_chatbot\data"

pdf_files = [
    file for file in os.listdir(pdf_folder)
    if file.lower().endswith(".pdf")
]

print("Total PDF files:", len(pdf_files))
print("Files:")

for file in pdf_files:
    print(file)

Total PDF files: 7
Files:
care--policy-usage-guide.pdf
care-freedom-(health-insurance-product)---brochure.pdf
care-freedom-(health-insurance-product)---prospectus-cum-sales-literature.pdf
care-opd---prospectus-cum-sales-literature.pdf
care-plus-(health-insurance-product)---brochure.pdf
care-plus--(health-insurance-product)---prospectus-cum-sales-literature.pdf
care-supreme----policy-usage-guide.pdf


## 2. PDF Text Extraction

In this step, I use PyMuPDF (`fitz`) to open the PDF documents
and extract text from each page.

The extracted text will be used for further preprocessing,
chunking, embedding, and retrieval.

In [2]:
import fitz

pdf_path = os.path.join(pdf_folder, pdf_files[0])

doc = fitz.open(pdf_path)

print("PDF:", pdf_files[0])
print("Number of pages:", len(doc))

PDF: care--policy-usage-guide.pdf
Number of pages: 4


In [3]:
text = ""

for page in doc:
    text += page.get_text()

print(text[:2000])

Policy Usage Guide
In-patient Hospitalization
Expenses incurred for treatment that requires you to stay hospitalised for more
than24 hours. It includes expenses on – room charges, nursing expenses and 
intensive care unit charges, surgeon's fee, doctor's fee, anesthesia, blood, 
oxygen, operation theater charges, etc.
Day Care Treatment
Refers to treatment that requires less than 24 Hours of hospitalisation. We cover 
more than 540 such treatments. Some of the examples are Surgery for Cataract, 
Cancer Chemotherapy, Haemodialysis etc.
Pre & Post-Hospitalization Medical Expenses
Medical expenses incurred on doctor’s consultation fee, prescribed diagnostics 
and prescribed medicines 30 days before date of admission and 60 days after date 
of discharge.
Domiciliary Hospitalization
Medical expenses incurred during his/her treatment at home so long as it involves 
medical treatment for a period exceeding 3 days. T&C apply.
Automatic Recharge
Reinstates up to the amount of Sum Insured, once 

## 3. Loading All Insurance Documents

Here, I load all 7 insurance PDFs and extract text from every page.

For each page, I store:
- Extracted text
- Source PDF name
- Page number

This creates a structured collection of the documents
for further processing.

In [4]:
all_documents = []

for filename in pdf_files:
    pdf_path = os.path.join(pdf_folder, filename)
    
    doc = fitz.open(pdf_path)
    
    for page_number, page in enumerate(doc):
        page_text = page.get_text()
        
        all_documents.append({
            "text": page_text,
            "source": filename,
            "page": page_number + 1
        })

print("Total pages loaded:", len(all_documents))
print("Total PDFs:", len(pdf_files))

Total pages loaded: 131
Total PDFs: 7


## 4. Adding Document Metadata

Metadata provides additional information about each document.

For this project, I add:
- Source file name
- Page number
- Document type

Document type metadata will later help in filtering documents
during the retrieval process.

In [5]:
document_types = {}

for filename in pdf_files:
    if "policy-usage-guide" in filename:
        document_type = "Policy Usage Guide"
    elif "brochure" in filename:
        document_type = "Brochure"
    elif "prospectus-cum-sales-literature" in filename:
        document_type = "Prospectus / Sales Literature"
    else:
        document_type = "Other"
    
    document_types[filename] = document_type

for filename, doc_type in document_types.items():
    print(filename, "->", doc_type)

care--policy-usage-guide.pdf -> Policy Usage Guide
care-freedom-(health-insurance-product)---brochure.pdf -> Brochure
care-freedom-(health-insurance-product)---prospectus-cum-sales-literature.pdf -> Prospectus / Sales Literature
care-opd---prospectus-cum-sales-literature.pdf -> Prospectus / Sales Literature
care-plus-(health-insurance-product)---brochure.pdf -> Brochure
care-plus--(health-insurance-product)---prospectus-cum-sales-literature.pdf -> Prospectus / Sales Literature
care-supreme----policy-usage-guide.pdf -> Policy Usage Guide


In [6]:
for document in all_documents:
    document["document_type"] = document_types[document["source"]]

print(all_documents[0])

{'text': "Policy Usage Guide\nIn-patient Hospitalization\nExpenses incurred for treatment that requires you to stay hospitalised for more\nthan24 hours. It includes expenses on – room charges, nursing expenses and \nintensive care unit charges, surgeon's fee, doctor's fee, anesthesia, blood, \noxygen, operation theater charges, etc.\nDay Care Treatment\nRefers to treatment that requires less than 24 Hours of hospitalisation. We cover \nmore than 540 such treatments. Some of the examples are Surgery for Cataract, \nCancer Chemotherapy, Haemodialysis etc.\nPre & Post-Hospitalization Medical Expenses\nMedical expenses incurred on doctor’s consultation fee, prescribed diagnostics \nand prescribed medicines 30 days before date of admission and 60 days after date \nof discharge.\nDomiciliary Hospitalization\nMedical expenses incurred during his/her treatment at home so long as it involves \nmedical treatment for a period exceeding 3 days. T&C apply.\nAutomatic Recharge\nReinstates up to the 

## 5. Text Cleaning

The extracted PDF text may contain unnecessary spaces,
line breaks, and formatting issues.

I clean the extracted text using regular expressions (`re`)
to make it more suitable for chunking and retrieval.

In [7]:
import re

def clean_text(text):
    
    text = re.sub(r'\s+', ' ', text)


    text = re.sub(r'([a-zA-Z])(\d)', r'\1 \2', text)
    text = re.sub(r'(\d)([a-zA-Z])', r'\1 \2', text)

    return text.strip()

for document in all_documents:
    document["clean_text"] = clean_text(document["text"])

print(all_documents[0]["clean_text"][:2000])

Policy Usage Guide In-patient Hospitalization Expenses incurred for treatment that requires you to stay hospitalised for more than 24 hours. It includes expenses on – room charges, nursing expenses and intensive care unit charges, surgeon's fee, doctor's fee, anesthesia, blood, oxygen, operation theater charges, etc. Day Care Treatment Refers to treatment that requires less than 24 Hours of hospitalisation. We cover more than 540 such treatments. Some of the examples are Surgery for Cataract, Cancer Chemotherapy, Haemodialysis etc. Pre & Post-Hospitalization Medical Expenses Medical expenses incurred on doctor’s consultation fee, prescribed diagnostics and prescribed medicines 30 days before date of admission and 60 days after date of discharge. Domiciliary Hospitalization Medical expenses incurred during his/her treatment at home so long as it involves medical treatment for a period exceeding 3 days. T&C apply. Automatic Recharge Reinstates up to the amount of Sum Insured, once during

## 6. Text Chunking

Large documents are divided into smaller chunks so that
the retrieval system can efficiently search for relevant information.

I use RecursiveCharacterTextSplitter with:
- Chunk size: 512 characters
- Chunk overlap: 100 characters

The overlap helps reduce information loss when relevant
information appears near a chunk boundary.

In [8]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=512,
    chunk_overlap=100
)

chunks = []

for document in all_documents:
    split_texts = text_splitter.split_text(document["clean_text"])
    
    for chunk in split_texts:
        chunks.append({
            "text": chunk,
            "source": document["source"],
            "page": document["page"],
            "document_type": document["document_type"]
        })

print("Total chunks:", len(chunks))
print("\nFirst chunk:")
print(chunks[0])

Total chunks: 872

First chunk:
{'text': "Policy Usage Guide In-patient Hospitalization Expenses incurred for treatment that requires you to stay hospitalised for more than 24 hours. It includes expenses on – room charges, nursing expenses and intensive care unit charges, surgeon's fee, doctor's fee, anesthesia, blood, oxygen, operation theater charges, etc. Day Care Treatment Refers to treatment that requires less than 24 Hours of hospitalisation. We cover more than 540 such treatments. Some of the examples are Surgery for Cataract, Cancer", 'source': 'care--policy-usage-guide.pdf', 'page': 1, 'document_type': 'Policy Usage Guide'}


## 7. Generating Embeddings

In this step, the document chunks are converted into numerical
vector representations using the `all-MiniLM-L6-v2` embedding model.

These embeddings help the system identify semantic similarity
between the user's query and the insurance document chunks.

The embeddings will later be stored in ChromaDB for retrieval.

In [9]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

print("Embedding model loaded successfully!")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding model loaded successfully!


In [10]:
texts = [chunk["text"] for chunk in chunks]

embeddings = embedding_model.encode(
    texts,
    show_progress_bar=True
)

print("Total embeddings:", len(embeddings))
print("Embedding dimension:", len(embeddings[0]))

Batches:   0%|          | 0/28 [00:00<?, ?it/s]

Total embeddings: 872
Embedding dimension: 384


## 8. Setting Up ChromaDB

ChromaDB is used as a vector database to store document chunks, their embeddings, and metadata.

It will allow us to efficiently retrieve relevant insurance document chunks when a user asks a question.

In [11]:
!pip install chromadb

In [12]:
import chromadb

client = chromadb.PersistentClient(path="./chroma_db")

collection = client.get_or_create_collection(
    name="insurance_documents"
)

print("ChromaDB collection created successfully!")

ChromaDB collection created successfully!


### 8.3 Storing Documents, Embeddings, and Metadata

The processed document chunks, generated embeddings, and metadata are stored in the ChromaDB collection.

Metadata such as source, page number, and document type will help in filtering and improving document retrieval accuracy.

In [13]:
documents = [chunk["text"] for chunk in chunks]

ids = [f"chunk_{i}" for i in range(len(chunks))]

metadatas = [
    {
        "source": chunk["source"],
        "page": chunk["page"],
        "document_type": chunk["document_type"]
    }
    for chunk in chunks
]

collection.add(
    ids=ids,
    documents=documents,
    embeddings=embeddings.tolist(),
    metadatas=metadatas
)

print("Documents stored successfully in ChromaDB!")
print("Total documents:", collection.count())

Documents stored successfully in ChromaDB!
Total documents: 872


## 9. Document Retrieval Using Similarity Search

In this step, a user query is converted into an embedding using the same embedding model used for the document chunks.

ChromaDB compares the query embedding with the stored document embeddings and retrieves the most relevant chunks.

This is the retrieval component of the RAG pipeline.

### 9.1 Creating a Test Query

A sample question related to the insurance documents is used to test whether the system can retrieve relevant information.

In [14]:
query = "What is the waiting period for pre-existing diseases?"

print("Query:", query)

Query: What is the waiting period for pre-existing diseases?


###  Generating Query Embedding

The user query is converted into a numerical vector using the same `all-MiniLM-L6-v2` embedding model.

In [15]:
query_embedding = embedding_model.encode(query).tolist()

print("Query embedding generated successfully!")
print("Embedding dimension:", len(query_embedding))

Query embedding generated successfully!
Embedding dimension: 384


###  Retrieving Relevant Document Chunks

The query embedding is searched against the ChromaDB collection.

The top 5 most relevant chunks are retrieved based on semantic similarity.

In [16]:
results = collection.query(
    query_embeddings=[query_embedding],
    n_results=5
)

print("Retrieved chunks:", len(results["documents"][0]))

Retrieved chunks: 5


In [17]:
for i, document in enumerate(results["documents"][0]):
    print(f"\n--- Result {i+1} ---")
    print(document[:500])


--- Result 1 ---
Illnesses, conditions or Pre-Existing Diseases at the time of commencement of first policy with Us, any Claim in respect of that Illness, condition or Pre-existing Disease shall not be covered until the completion of 24 months of continuous insurance coverage with Us from the first Policy Period Start Date. 3. Pre-existing Disease: (Code - Excl 01) a. Expenses related to the treatment of a pre-existing Disease (PED) and its direct complications shall be excluded until the expiry of 24 months of 

--- Result 2 ---
Insurance) Regulations, then waiting period for the same would be reduced to the extent of prior coverage. d. Coverage under the policy after the expiry of 24 months for any pre-existing disease is subject to the same being declared at the time of application and accepted by Insurer. 4. The Waiting Periods as defined in Clauses 4.1(a), 4.1(b) and 4.1(c) of terms and conditions shall be applicable individually for each Insured Person and Claims shall be assess

##  Metadata Filtering

Since the dataset contains different types of insurance documents, metadata filtering is used to restrict retrieval to a specific document type.

This helps improve retrieval relevance and accuracy by searching only within the required category of documents.

### Filtering by Document Type

The retrieval system is tested by filtering the search results to documents classified as `Policy Usage Guide`.

In [18]:
filtered_results = collection.query(
    query_embeddings=[query_embedding],
    n_results=5,
    where={"document_type": "Policy Usage Guide"}
)

print("Retrieved chunks:", len(filtered_results["documents"][0]))

Retrieved chunks: 5


In [19]:
for i, document in enumerate(filtered_results["documents"][0]):
    print(f"\n--- Filtered Result {i+1} ---")
    print(document[:500])


--- Filtered Result 1 ---
provided is representative and summary of waiting period applicable under the policy, for complete details please refer to T&C WAITING PERIOD The time span during which you can not claim some or all benefits, specified for the illnesses covered under the insurance policy. A 30 day initial wait period applies at start of policy except for policy renewal & accident cases. For specified illness, treatment & surgeries for example – Hernia, Ulcer, ENT related, Arthirities, Ligament Tear, Cataract & S

--- Filtered Result 2 ---
for example – Hernia, Ulcer, ENT related, Arthirities, Ligament Tear, Cataract & Stone the applicant will wait 24 month and for pre-existing aliment 36 months. Clause 4.1(a) (i), (ii), (iii)

--- Filtered Result 3 ---
PED waiting period related to Diabetes/ Hypertension/ Hyperlipidemia/ Asthma, at the time of issuance of first policy with the Company. Co-payment This optional Benefit will give an option to bear a Co-payment, as specified in 

### Verifying Retrieved Metadata

The metadata of the retrieved chunks is checked to confirm that all results belong to the requested document type.

In [20]:
for i, metadata in enumerate(filtered_results["metadatas"][0]):
    print(f"Result {i+1}:")
    print(metadata)

Result 1:
{'page': 3, 'source': 'care-supreme----policy-usage-guide.pdf', 'document_type': 'Policy Usage Guide'}
Result 2:
{'source': 'care-supreme----policy-usage-guide.pdf', 'document_type': 'Policy Usage Guide', 'page': 3}
Result 3:
{'source': 'care-supreme----policy-usage-guide.pdf', 'page': 3, 'document_type': 'Policy Usage Guide'}
Result 4:
{'document_type': 'Policy Usage Guide', 'page': 2, 'source': 'care-supreme----policy-usage-guide.pdf'}
Result 5:
{'source': 'care--policy-usage-guide.pdf', 'page': 1, 'document_type': 'Policy Usage Guide'}


## Setting Up the Local LLM

In this step, a local Large Language Model (LLM) is configured for generating answers from the retrieved insurance document context.

A local model is used to avoid API costs during development and testing.

In [21]:
!pip install langchain-ollama

In [22]:
from langchain_ollama import ChatOllama

llm = ChatOllama(
    model="llama3.2",
    temperature=0
)

print("Local LLM configured successfully!")

Local LLM configured successfully!


### Testing the LLM

A simple test question is sent to the local LLM to verify that the model is working correctly before integrating it with the RAG retrieval pipeline.

In [23]:
response = llm.invoke("What is health insurance?")

print(response.content)

Health insurance is a type of insurance that helps cover the cost of medical expenses, including doctor visits, hospital stays, surgeries, and other healthcare services. It provides financial protection against the high costs of medical care, allowing individuals and families to access necessary healthcare services without breaking the bank.

Here's how health insurance typically works:

1. **Premiums**: Policyholders pay a monthly or annual premium to the insurance company in exchange for coverage.
2. **Coverage**: The insurance company agrees to pay a portion of the medical expenses incurred by the policyholder, usually up to a certain limit or percentage of the total cost.
3. **Network**: Most health insurance plans have a network of healthcare providers who have agreed to provide services to policyholders at a discounted rate.
4. **Claims**: When a policyholder receives medical care, they submit a claim to the insurance company, which reviews and pays the eligible expenses.

Types 

##  Connecting Retrieval with the LLM

In this step, the user's question is converted into an embedding and used to retrieve the most relevant document chunks from ChromaDB.

The retrieved chunks are then combined to create the context that will be provided to the LLM.

In [24]:
query = "What is the waiting period for pre-existing diseases?"
query_embedding = embedding_model.encode(query).tolist()
results = collection.query(
    query_embeddings=[query_embedding],
    n_results=5
)
retrieved_docs = results["documents"][0]
context = "\n\n".join(retrieved_docs)

print("Query:")
print(query)

print("\nNumber of retrieved chunks:", len(retrieved_docs))

print("\nRetrieved Context:")
print(context[:3000])

Query:
What is the waiting period for pre-existing diseases?

Number of retrieved chunks: 5

Retrieved Context:
Illnesses, conditions or Pre-Existing Diseases at the time of commencement of first policy with Us, any Claim in respect of that Illness, condition or Pre-existing Disease shall not be covered until the completion of 24 months of continuous insurance coverage with Us from the first Policy Period Start Date. 3. Pre-existing Disease: (Code - Excl 01) a. Expenses related to the treatment of a pre-existing Disease (PED) and its direct complications shall be excluded until the expiry of 24 months of continuous

Insurance) Regulations, then waiting period for the same would be reduced to the extent of prior coverage. d. Coverage under the policy after the expiry of 24 months for any pre-existing disease is subject to the same being declared at the time of application and accepted by Insurer. 4. The Waiting Periods as defined in Clauses 4.1(a), 4.1(b) and 4.1(c) of terms and conditi

## 13. Creating the RAG Prompt

The retrieved document context is combined with the user's question to create a prompt for the LLM.

The LLM is instructed to answer the question using only the information available in the retrieved insurance documents.

In [25]:
prompt = f"""
You are an insurance information assistant.

Answer the question using only the information provided in the context.

If the answer is not available in the context, say:
"I could not find the answer in the provided documents."

Context:
{context}

Question:
{query}

Answer:
"""

print(prompt[:4000])


You are an insurance information assistant.

Answer the question using only the information provided in the context.

If the answer is not available in the context, say:
"I could not find the answer in the provided documents."

Context:
Illnesses, conditions or Pre-Existing Diseases at the time of commencement of first policy with Us, any Claim in respect of that Illness, condition or Pre-existing Disease shall not be covered until the completion of 24 months of continuous insurance coverage with Us from the first Policy Period Start Date. 3. Pre-existing Disease: (Code - Excl 01) a. Expenses related to the treatment of a pre-existing Disease (PED) and its direct complications shall be excluded until the expiry of 24 months of continuous

Insurance) Regulations, then waiting period for the same would be reduced to the extent of prior coverage. d. Coverage under the policy after the expiry of 24 months for any pre-existing disease is subject to the same being declared at the time of ap

## 14. Generating the Final RAG Answer

The RAG prompt containing the user's question and the retrieved document context is sent to the local Llama 3.2 model.

The model generates the final answer based on the retrieved insurance information.

In [26]:
response = llm.invoke(prompt)

print("Question:")
print(query)

print("\nFinal Answer:")
print(response.content)

Question:
What is the waiting period for pre-existing diseases?

Final Answer:
The waiting period for pre-existing diseases is 24 months of continuous insurance coverage.


## 15. Displaying Retrieved Sources

The source file, page number, and document type of the retrieved chunks are displayed.

This helps verify which insurance documents were used during the retrieval process.

In [27]:
print("Retrieved Sources:\n")

for i, metadata in enumerate(results["metadatas"][0]):
    print(f"Result {i+1}:")
    print("Source:", metadata["source"])
    print("Page:", metadata["page"])
    print("Document Type:", metadata["document_type"])
    print()

Retrieved Sources:

Result 1:
Source: care-freedom-(health-insurance-product)---prospectus-cum-sales-literature.pdf
Page: 12
Document Type: Prospectus / Sales Literature

Result 2:
Source: care-freedom-(health-insurance-product)---prospectus-cum-sales-literature.pdf
Page: 12
Document Type: Prospectus / Sales Literature

Result 3:
Source: care-plus--(health-insurance-product)---prospectus-cum-sales-literature.pdf
Page: 12
Document Type: Prospectus / Sales Literature

Result 4:
Source: care-freedom-(health-insurance-product)---prospectus-cum-sales-literature.pdf
Page: 11
Document Type: Prospectus / Sales Literature

Result 5:
Source: care-supreme----policy-usage-guide.pdf
Page: 3
Document Type: Policy Usage Guide



In [28]:
def ask_rag(query):
    # Query ko embedding mein convert karna
    query_embedding = embedding_model.encode(query).tolist()

    # ChromaDB se top 5 relevant chunks retrieve karna
    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=5
    )

    retrieved_docs = results["documents"][0]
    context = "\n\n".join(retrieved_docs)

    # RAG prompt
    prompt = f"""
You are an insurance information assistant.

Answer the question using only the information provided in the context.

If the answer is not available in the context, say:
"I could not find the answer in the provided documents."

Context:
{context}

Question:
{query}

Answer:
"""

    # LLM se answer generate karna
    response = llm.invoke(prompt)

    return response.content

In [29]:
questions = [
    "What is the waiting period for pre-existing diseases?",
    "What is covered under the health insurance policy?",
    "What are the eligibility criteria for the policy?",
    "What is the claim process?",
    "What are the exclusions under the policy?"
]

for question in questions:
    answer = ask_rag(question)

    print("\n" + "=" * 70)
    print("QUESTION:", question)
    print("=" * 70)
    print("ANSWER:", answer)


QUESTION: What is the waiting period for pre-existing diseases?
ANSWER: The waiting period for pre-existing diseases is 24 months of continuous insurance coverage.

QUESTION: What is covered under the health insurance policy?
ANSWER: According to the policy, the following is covered under the health insurance policy:

* Reasonable and Customary Charges incurred towards medical treatment taken or any other expenses triggered under any Benefit during the Policy Period (as stated in section f of the policy).
* Room charges, nursing expenses, intensive care unit charges, surgeon's fee, and doctor's fees for in-patient treatment, if the Insured Person is hospitalized for a minimum period of 24 consecutive hours (as stated in section 11 of the policy).
* Physical consultations with General Physicians (as stated in section 3.1.1 of the policy).
* Optional coverage, including Allopathic line of treatment, unless alternate line of treatment is specifically covered under the Base Benefit or Opt

###  Baseline Retrieval

The baseline retrieval system uses semantic similarity search to retrieve the most relevant document chunks from ChromaDB. The user query is converted into an embedding using the all-MiniLM-L6-v2 model, and the top-k relevant chunks are retrieved from the vector database.

In [30]:
# Baseline Retrieval Function

def retrieve_documents(query, k=5):
    # Convert the query into an embedding
    query_embedding = embedding_model.encode(query).tolist()

    # Retrieve top-k relevant chunks from ChromaDB
    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=k
    )

    return results

print("Baseline retrieval function created successfully!")

Baseline retrieval function created successfully!


###  Top-k Retrieval Evaluation

The retrieval performance is tested using different values of k. The value of k determines the number of document chunks retrieved for each user query. In this experiment, k values of 3, 5, and 10 are compared to observe the effect of the number of retrieved chunks on the available context.

In [31]:
# Top-k Retrieval Experiment

test_query = "What is the waiting period for pre-existing diseases?"

k_values = [3, 5, 10]

for k in k_values:
    results = retrieve_documents(test_query, k=k)

    print("\n" + "=" * 70)
    print(f"TOP-K = {k}")
    print("=" * 70)

    for i, doc in enumerate(results["documents"][0]):
        print(f"\n--- Retrieved Chunk {i+1} ---")
        print(doc[:300])


TOP-K = 3

--- Retrieved Chunk 1 ---
Illnesses, conditions or Pre-Existing Diseases at the time of commencement of first policy with Us, any Claim in respect of that Illness, condition or Pre-existing Disease shall not be covered until the completion of 24 months of continuous insurance coverage with Us from the first Policy Period Sta

--- Retrieved Chunk 2 ---
Insurance) Regulations, then waiting period for the same would be reduced to the extent of prior coverage. d. Coverage under the policy after the expiry of 24 months for any pre-existing disease is subject to the same being declared at the time of application and accepted by Insurer. 4. The Waiting 

--- Retrieved Chunk 3 ---
afresh to the extent of sum insured increase. c. If the Insured Person is continuously covered without any break as defined under the portability norms of the extant IRDAI (Health Insurance) Regulations, then waiting period for the same would be reduced to the extent of prior coverage. d. Coverage u

TOP-

###  BM25 Retrieval

BM25 (Best Matching 25) is a keyword-based information retrieval technique. Unlike semantic search, which uses embeddings to understand meaning, BM25 retrieves documents based on the occurrence and importance of query terms. In this project, BM25 retrieval is implemented as an additional retrieval method for comparison with semantic vector search.

In [32]:
!pip install rank-bm25

In [33]:
from rank_bm25 import BM25Okapi

# All document chunks
corpus = [chunk["text"] for chunk in chunks]

# Tokenize the chunks
tokenized_corpus = [doc.lower().split() for doc in corpus]

# Create BM25 index
bm25 = BM25Okapi(tokenized_corpus)

print("BM25 index created successfully!")
print("Total documents indexed:", len(corpus))

BM25 index created successfully!
Total documents indexed: 872


In [34]:
query = "What is the waiting period for pre-existing diseases?"

# Tokenize query
tokenized_query = query.lower().split()

# Get BM25 scores
scores = bm25.get_scores(tokenized_query)

# Get top 5 document indexes
top_n = sorted(
    range(len(scores)),
    key=lambda i: scores[i],
    reverse=True
)[:5]

print("BM25 Top 5 Retrieved Chunks:")

for rank, index in enumerate(top_n, start=1):
    print(f"\n--- Result {rank} ---")
    print("Score:", round(scores[index], 4))
    print(corpus[index][:500])

BM25 Top 5 Retrieved Chunks:

--- Result 1 ---
Score: 17.3868
apply afresh to the extent of sum insured increase. c. If any of the specified disease/ procedure falls under the waiting period specified for pre-Existing diseases, then the longer of the two waiting periods shall apply. d. The waiting period for listed conditions shall apply even if contracted after the policy or declared and accepted without a specific exclusion. e. If the Insured Person is continuously covered without any break as defined under the applicable norms on portability stipulated 

--- Result 2 ---
Score: 17.3205
b. In case of enhancement of sum insured the exclusion shall apply afresh to the extent of sum insured increase. c. If any of the specified disease/procedure falls under the waiting period specified for pre-Existing diseases, then the longer of the two waiting periods shall apply. d. The waiting period for listed conditions shall apply even if contracted after the policy or declared and accepted witho

###  Hybrid Retrieval: BM25 + Semantic Search

Hybrid retrieval combines keyword-based BM25 retrieval with semantic vector search. BM25 helps retrieve documents containing important query terms, while semantic search retrieves documents based on contextual similarity. Combining both approaches can improve retrieval performance, especially when the dataset contains similar or overlapping insurance documents.

In [35]:
# Hybrid Retrieval: BM25 + Semantic Search

def hybrid_retrieval(query, k=5):
    # -----------------------------
    # 1. BM25 Retrieval
    # -----------------------------
    tokenized_query = query.lower().split()
    bm25_scores = bm25.get_scores(tokenized_query)

    bm25_top_n = sorted(
        range(len(bm25_scores)),
        key=lambda i: bm25_scores[i],
        reverse=True
    )[:k]

    # -----------------------------
    # 2. Semantic Retrieval
    # -----------------------------
    query_embedding = embedding_model.encode(query).tolist()

    semantic_results = collection.query(
        query_embeddings=[query_embedding],
        n_results=k
    )

    semantic_docs = semantic_results["documents"][0]

    # -----------------------------
    # 3. Combine Results
    # -----------------------------
    hybrid_docs = []

    # Add BM25 results
    for index in bm25_top_n:
        hybrid_docs.append(corpus[index])

    # Add semantic results if not already present
    for doc in semantic_docs:
        if doc not in hybrid_docs:
            hybrid_docs.append(doc)

    # Return top k combined documents
    return hybrid_docs[:k]


# Test Hybrid Retrieval

test_query = "What is the waiting period for pre-existing diseases?"

hybrid_results = hybrid_retrieval(test_query, k=5)

print("Hybrid Retrieval - Top 5 Results")
print("=" * 70)

for i, doc in enumerate(hybrid_results, start=1):
    print(f"\n--- Result {i} ---")
    print(doc[:500])

Hybrid Retrieval - Top 5 Results

--- Result 1 ---
apply afresh to the extent of sum insured increase. c. If any of the specified disease/ procedure falls under the waiting period specified for pre-Existing diseases, then the longer of the two waiting periods shall apply. d. The waiting period for listed conditions shall apply even if contracted after the policy or declared and accepted without a specific exclusion. e. If the Insured Person is continuously covered without any break as defined under the applicable norms on portability stipulated 

--- Result 2 ---
b. In case of enhancement of sum insured the exclusion shall apply afresh to the extent of sum insured increase. c. If any of the specified disease/procedure falls under the waiting period specified for pre-Existing diseases, then the longer of the two waiting periods shall apply. d. The waiting period for listed conditions shall apply even if contracted after the policy or declared and accepted without a specific exclusion. e

### Reciprocal Rank Fusion (RRF)

Reciprocal Rank Fusion (RRF) is a ranking technique used to combine results from multiple retrieval methods. In this project, RRF combines the rankings produced by BM25 keyword retrieval and semantic vector retrieval. Documents appearing highly in both retrieval lists receive a higher combined score, which can improve the final ranking of relevant chunks.

In [36]:
# Reciprocal Rank Fusion (RRF)

def rrf_retrieval(query, k=5, rrf_k=60):
    # -----------------------------
    # 1. BM25 Retrieval
    # -----------------------------
    tokenized_query = query.lower().split()

    bm25_scores = bm25.get_scores(tokenized_query)

    bm25_ranked = sorted(
        range(len(bm25_scores)),
        key=lambda i: bm25_scores[i],
        reverse=True
    )[:k]

    # -----------------------------
    # 2. Semantic Retrieval
    # -----------------------------
    query_embedding = embedding_model.encode(query).tolist()

    semantic_results = collection.query(
        query_embeddings=[query_embedding],
        n_results=k
    )

    semantic_docs = semantic_results["documents"][0]

    # Find indexes of semantic documents
    semantic_ranked = []

    for doc in semantic_docs:
        if doc in corpus:
            semantic_ranked.append(corpus.index(doc))

    # -----------------------------
    # 3. Calculate RRF Scores
    # -----------------------------
    rrf_scores = {}

    for rank, index in enumerate(bm25_ranked, start=1):
        rrf_scores[index] = rrf_scores.get(index, 0) + 1 / (rrf_k + rank)

    for rank, index in enumerate(semantic_ranked, start=1):
        rrf_scores[index] = rrf_scores.get(index, 0) + 1 / (rrf_k + rank)

    # -----------------------------
    # 4. Sort by RRF Score
    # -----------------------------
    ranked_results = sorted(
        rrf_scores.items(),
        key=lambda x: x[1],
        reverse=True
    )[:k]

    return ranked_results


# Test RRF Retrieval

test_query = "What is the waiting period for pre-existing diseases?"

rrf_results = rrf_retrieval(test_query, k=5)

print("RRF Retrieval - Top 5 Results")
print("=" * 70)

for rank, (index, score) in enumerate(rrf_results, start=1):
    print(f"\n--- Result {rank} ---")
    print("RRF Score:", round(score, 6))
    print(corpus[index][:500])

RRF Retrieval - Top 5 Results

--- Result 1 ---
RRF Score: 0.031754
Insurance) Regulations, then waiting period for the same would be reduced to the extent of prior coverage. d. Coverage under the policy after the expiry of 24 months for any pre-existing disease is subject to the same being declared at the time of application and accepted by Insurer. 4. The Waiting Periods as defined in Clauses 4.1(a), 4.1(b) and 4.1(c) of terms and conditions shall be applicable individually for each Insured Person and Claims shall be assessed accordingly. 5. If Coverage for Be

--- Result 2 ---
RRF Score: 0.031258
afresh to the extent of sum insured increase. c. If the Insured Person is continuously covered without any break as defined under the portability norms of the extant IRDAI (Health Insurance) Regulations, then waiting period for the same would be reduced to the extent of prior coverage. d. Coverage under the policy after the expiry of 36 months for any pre-existing disease is subject to the 

###  Retrieval Evaluation Dataset

A small evaluation dataset is created to measure the retrieval performance of the RAG system. Each test question is associated with relevant document content that should ideally be retrieved by the system. Precision, recall, and F1-score are then used to evaluate retrieval quality.

In [37]:
# Retrieval Evaluation Dataset

evaluation_questions = [
    {
        "question": "What is the waiting period for pre-existing diseases?",
        "expected_document_type": "Policy Usage Guide"
    },
    {
        "question": "What are the exclusions under the health insurance policy?",
        "expected_document_type": "Policy Usage Guide"
    },
    {
        "question": "What are the benefits provided under the Care Plus health insurance plan?",
        "expected_document_type": "Brochure"
    },
    {
        "question": "What are the features of the Care Freedom health insurance product?",
        "expected_document_type": "Brochure"
    },
    {
        "question": "What is the claim process for the health insurance policy?",
        "expected_document_type": "Policy Usage Guide"
    }
]

print("Evaluation dataset created successfully!")
print("Total evaluation questions:", len(evaluation_questions))

for i, item in enumerate(evaluation_questions, start=1):
    print(f"\n{i}. {item['question']}")
    print("Expected Document Type:", item["expected_document_type"])

Evaluation dataset created successfully!
Total evaluation questions: 5

1. What is the waiting period for pre-existing diseases?
Expected Document Type: Policy Usage Guide

2. What are the exclusions under the health insurance policy?
Expected Document Type: Policy Usage Guide

3. What are the benefits provided under the Care Plus health insurance plan?
Expected Document Type: Brochure

4. What are the features of the Care Freedom health insurance product?
Expected Document Type: Brochure

5. What is the claim process for the health insurance policy?
Expected Document Type: Policy Usage Guide


In [38]:
# Retrieval Evaluation: Precision, Recall and F1

def calculate_metrics(retrieved_indices, expected_document_type):
    # Total relevant chunks in the dataset
    relevant_indices = {
        i for i, chunk in enumerate(chunks)
        if chunk["document_type"] == expected_document_type
    }

    # Retrieved chunks that belong to the expected document type
    retrieved_set = set(retrieved_indices)
    relevant_retrieved = retrieved_set.intersection(relevant_indices)

    # Precision
    precision = (
        len(relevant_retrieved) / len(retrieved_set)
        if len(retrieved_set) > 0 else 0
    )

    # Recall
    recall = (
        len(relevant_retrieved) / len(relevant_indices)
        if len(relevant_indices) > 0 else 0
    )

    # F1 Score
    if precision + recall > 0:
        f1 = 2 * (precision * recall) / (precision + recall)
    else:
        f1 = 0
    return precision, recall, f1

   

# Evaluate Baseline, BM25, Hybrid and RRF

results_table = []

for item in evaluation_questions:
    query = item["question"]
    expected_type = item["expected_document_type"]

    # -----------------------------
    # Baseline Semantic Retrieval
    # -----------------------------
    semantic_results = retrieve_documents(query, k=5)

    semantic_indices = []

    for doc in semantic_results["documents"][0]:
        if doc in corpus:
            semantic_indices.append(corpus.index(doc))

    p, r, f1 = calculate_metrics(
        semantic_indices,
        expected_type
    )

    results_table.append([
        query,
        "Baseline Semantic",
        p,
        r,
        f1
    ])

    # -----------------------------
    # BM25 Retrieval
    # -----------------------------
    tokenized_query = query.lower().split()
    bm25_scores = bm25.get_scores(tokenized_query)

    bm25_indices = sorted(
        range(len(bm25_scores)),
        key=lambda i: bm25_scores[i],
        reverse=True
    )[:5]

    p, r, f1 = calculate_metrics(
        bm25_indices,
        expected_type
    )

    results_table.append([
        query,
        "BM25",
        p,
        r,
        f1
    ])

    # -----------------------------
    # RRF Retrieval
    # -----------------------------
    rrf_results = rrf_retrieval(query, k=5)

    rrf_indices = [index for index, score in rrf_results]

    p, r, f1 = calculate_metrics(
        rrf_indices,
        expected_type
    )

    results_table.append([
        query,
        "RRF",
        p,
        r,
        f1
    ])


print("Retrieval evaluation completed!")
print("\nQuestion-wise Results:\n")

for row in results_table:
    print("=" * 70)
    print("Question:", row[0])
    print("Method:", row[1])
    print("Precision:", round(row[2], 4))
    print("Recall:", round(row[3], 4))
    print("F1 Score:", round(row[4], 4))

Retrieval evaluation completed!

Question-wise Results:

Question: What is the waiting period for pre-existing diseases?
Method: Baseline Semantic
Precision: 0.2
Recall: 0.0227
F1 Score: 0.0408
Question: What is the waiting period for pre-existing diseases?
Method: BM25
Precision: 0.2
Recall: 0.0227
F1 Score: 0.0408
Question: What is the waiting period for pre-existing diseases?
Method: RRF
Precision: 0.0
Recall: 0.0
F1 Score: 0
Question: What are the exclusions under the health insurance policy?
Method: Baseline Semantic
Precision: 0.0
Recall: 0.0
F1 Score: 0
Question: What are the exclusions under the health insurance policy?
Method: BM25
Precision: 0.0
Recall: 0.0
F1 Score: 0
Question: What are the exclusions under the health insurance policy?
Method: RRF
Precision: 0.0
Recall: 0.0
F1 Score: 0
Question: What are the benefits provided under the Care Plus health insurance plan?
Method: Baseline Semantic
Precision: 0.2
Recall: 0.0172
F1 Score: 0.0317
Question: What are the benefits pro

query = "What is the waiting period for pre-existing diseases?"

hybrid_docs = hybrid_retrieval(query, k=5)

for i, doc in enumerate(hybrid_docs):
    print(f"\nResult {i+1}:")
    print(doc)

In [39]:
print("chunks:", len(chunks))
print("corpus:", len(corpus))
print("evaluation questions:", len(evaluation_questions))

chunks: 872
corpus: 872
evaluation questions: 5


In [40]:
# =====================================================
# COMPLETE RETRIEVAL EVALUATION
# Baseline + BM25 + Hybrid + RRF
# =====================================================

results_table = []

for item in evaluation_questions:

    query = item["question"]
    expected_type = item["expected_document_type"]

    # -------------------------------------------------
    # 1. BASELINE SEMANTIC RETRIEVAL
    # -------------------------------------------------

    semantic_results = retrieve_documents(
        query,
        k=5
    )

    semantic_indices = []

    for doc in semantic_results["documents"][0]:

        if doc in corpus:
            semantic_indices.append(
                corpus.index(doc)
            )

    p, r, f1 = calculate_metrics(
        semantic_indices,
        expected_type
    )

    results_table.append([
        query,
        "Baseline Semantic",
        p,
        r,
        f1
    ])


    # -------------------------------------------------
    # 2. BM25 RETRIEVAL
    # -------------------------------------------------

    tokenized_query = query.lower().split()

    bm25_scores = bm25.get_scores(
        tokenized_query
    )

    bm25_indices = sorted(
        range(len(bm25_scores)),
        key=lambda i: bm25_scores[i],
        reverse=True
    )[:5]

    p, r, f1 = calculate_metrics(
        bm25_indices,
        expected_type
    )

    results_table.append([
        query,
        "BM25",
        p,
        r,
        f1
    ])


    # -------------------------------------------------
    # 3. HYBRID BM25 + SEMANTIC
    # -------------------------------------------------

    hybrid_docs = hybrid_retrieval(
        query,
        k=5
    )

    hybrid_indices = []

    for doc in hybrid_docs:

        if doc in corpus:
            hybrid_indices.append(
                corpus.index(doc)
            )

    p, r, f1 = calculate_metrics(
        hybrid_indices,
        expected_type
    )

    results_table.append([
        query,
        "Hybrid BM25 + Semantic",
        p,
        r,
        f1
    ])


    # -------------------------------------------------
    # 4. RRF
    # -------------------------------------------------

    rrf_results = rrf_retrieval(
        query,
        k=5
    )

    rrf_indices = [
        index
        for index, score in rrf_results
    ]

    p, r, f1 = calculate_metrics(
        rrf_indices,
        expected_type
    )

    results_table.append([
        query,
        "RRF",
        p,
        r,
        f1
    ])


print("Retrieval evaluation completed!")

Retrieval evaluation completed!


In [41]:
import pandas as pd

evaluation_df = pd.DataFrame(
    results_table,
    columns=[
        "Question",
        "Method",
        "Precision",
        "Recall",
        "F1 Score"
    ]
)

comparison_df = (
    evaluation_df
    .groupby("Method")[[
        "Precision",
        "Recall",
        "F1 Score"
    ]]
    .mean()
    .reset_index()
)

comparison_df[[
    "Precision",
    "Recall",
    "F1 Score"
]] = comparison_df[[
    "Precision",
    "Recall",
    "F1 Score"
]].round(4)

print("Average Retrieval Performance")

display(comparison_df)

Average Retrieval Performance


,Method,Precision,Recall,F1 Score
0,BM25,0.20,0.0183,0.0336
1,Baseline Semantic,0.16,0.0149,0.0272
2,Hybrid BM25 + Semantic,0.20,0.0183,0.0336
3,RRF,0.20,0.0172,0.0317


In [42]:
# =====================================================
# TOP-K EVALUATION
# Compare k = 3, 5, 10
# =====================================================

top_k_results = []

for k in [3, 5, 10]:

    for item in evaluation_questions:

        query = item["question"]
        expected_type = item["expected_document_type"]

        # Semantic retrieval
        semantic_results = retrieve_documents(query, k=k)

        semantic_indices = []

        for doc in semantic_results["documents"][0]:
            if doc in corpus:
                semantic_indices.append(corpus.index(doc))

        p, r, f1 = calculate_metrics(
            semantic_indices,
            expected_type
        )

        top_k_results.append([
            k,
            query,
            p,
            r,
            f1
        ])

print("Top-k evaluation completed!")

Top-k evaluation completed!


In [43]:
# =====================================================
# TOP-K AVERAGE COMPARISON
# =====================================================

top_k_df = pd.DataFrame(
    top_k_results,
    columns=[
        "K",
        "Question",
        "Precision",
        "Recall",
        "F1 Score"
    ]
)

top_k_comparison = (
    top_k_df
    .groupby("K")[[
        "Precision",
        "Recall",
        "F1 Score"
    ]]
    .mean()
    .reset_index()
)

top_k_comparison[[
    "Precision",
    "Recall",
    "F1 Score"
]] = top_k_comparison[[
    "Precision",
    "Recall",
    "F1 Score"
]].round(4)

print("Top-k Average Performance")

display(top_k_comparison)

Top-k Average Performance


,K,Precision,Recall,F1 Score
0,3,0.20,0.0103,0.0197
1,5,0.16,0.0149,0.0272
2,10,0.12,0.0218,0.0368


In [44]:
final_comparison = comparison_df.copy()

# Top-k = 10 result
top_k_10 = top_k_comparison[
    top_k_comparison["K"] == 10
].copy()

top_k_10["Method"] = "Top-k = 10"

top_k_10 = top_k_10[
    ["Method", "Precision", "Recall", "F1 Score"]
]

# Existing methods
final_methods = final_comparison[
    ["Method", "Precision", "Recall", "F1 Score"]
]

# Combine
final_results = pd.concat(
    [final_methods, top_k_10],
    ignore_index=True
)

print("Final Retrieval Performance")

display(final_results)

Final Retrieval Performance


,Method,Precision,Recall,F1 Score
0,BM25,0.20,0.0183,0.0336
1,Baseline Semantic,0.16,0.0149,0.0272
2,Hybrid BM25 + Semantic,0.20,0.0183,0.0336
3,RRF,0.20,0.0172,0.0317
4,Top-k = 10,0.12,0.0218,0.0368


In [45]:
evaluation_questions = [
    {
        "question": "What is the waiting period for pre-existing diseases?",
        "expected_document_type": "Policy Usage Guide"
    },
    {
        "question": "What are the exclusions under the health insurance policy?",
        "expected_document_type": "Policy Usage Guide"
    },
    {
        "question": "What are the benefits provided under the Care Plus health insurance plan?",
        "expected_document_type": "Brochure"
    },
    {
        "question": "What are the features of the Care Freedom health insurance product?",
        "expected_document_type": "Brochure"
    },
    {
        "question": "What is the claim process for the health insurance policy?",
        "expected_document_type": "Policy Usage Guide"
    }
]

print("Total evaluation questions:", len(evaluation_questions))

for i, item in enumerate(evaluation_questions, start=1):
    print(f"{i}. {item['question']}")
    print("   Expected document type:", item["expected_document_type"])

Total evaluation questions: 5
1. What is the waiting period for pre-existing diseases?
   Expected document type: Policy Usage Guide
2. What are the exclusions under the health insurance policy?
   Expected document type: Policy Usage Guide
3. What are the benefits provided under the Care Plus health insurance plan?
   Expected document type: Brochure
4. What are the features of the Care Freedom health insurance product?
   Expected document type: Brochure
5. What is the claim process for the health insurance policy?
   Expected document type: Policy Usage Guide


In [46]:
question = "What is the waiting period for pre-existing diseases?"

query_embedding = embedding_model.encode(question).tolist()

results = collection.query(
    query_embeddings=[query_embedding],
    n_results=5,
    where={"document_type": "Policy Usage Guide"}
)

for i, document in enumerate(results["documents"][0], start=1):
    metadata = results["metadatas"][0][i - 1]

    print(f"\n--- Retrieved Chunk {i} ---")
    print("Source:", metadata["source"])
    print("Page:", metadata["page"])
    print("Document Type:", metadata["document_type"])
    print("Text:", document)


--- Retrieved Chunk 1 ---
Source: care-supreme----policy-usage-guide.pdf
Page: 3
Document Type: Policy Usage Guide
Text: provided is representative and summary of waiting period applicable under the policy, for complete details please refer to T&C WAITING PERIOD The time span during which you can not claim some or all benefits, specified for the illnesses covered under the insurance policy. A 30 day initial wait period applies at start of policy except for policy renewal & accident cases. For specified illness, treatment & surgeries for example – Hernia, Ulcer, ENT related, Arthirities, Ligament Tear, Cataract & Stone the

--- Retrieved Chunk 2 ---
Source: care-supreme----policy-usage-guide.pdf
Page: 3
Document Type: Policy Usage Guide
Text: for example – Hernia, Ulcer, ENT related, Arthirities, Ligament Tear, Cataract & Stone the applicant will wait 24 month and for pre-existing aliment 36 months. Clause 4.1(a) (i), (ii), (iii)

--- Retrieved Chunk 3 ---
Source: care-supreme----polic

In [47]:
question = "What are the exclusions under the health insurance policy?"

query_embedding = embedding_model.encode(question).tolist()

results = collection.query(
    query_embeddings=[query_embedding],
    n_results=5,
    where={"document_type": "Policy Usage Guide"}
)

for i, document in enumerate(results["documents"][0], start=1):
    metadata = results["metadatas"][0][i - 1]
    print(f"\n--- Retrieved Chunk {i} ---")
    print("Source:", metadata["source"])
    print("Page:", metadata["page"])
    print("Document Type:", metadata["document_type"])
    print("Text:", document)


--- Retrieved Chunk 1 ---
Source: care--policy-usage-guide.pdf
Page: 3
Document Type: Policy Usage Guide
Text: A mandatory Co-Payment of 10% per claim is applicable. Clause 3.2.8 Clause 3.2.14 Clause 3.1.11 WHAT IS NOT COVERED? Any hospital admission primarily for investigation/diagnostic purposes, infertility, substance abuse, self-inflicted injuries, war, civil war or breach of law. Treatment expenses in blacklisted hospitals is also not covered. Note:-The availability of benefits depends upon the selected plan and the sum insured Clause 4.1(b)& 4.2 Clause 3.2.1 There are two modes of claiming, cashless &

--- Retrieved Chunk 2 ---
Source: care--policy-usage-guide.pdf
Page: 2
Document Type: Policy Usage Guide
Text: for 68 Non-Payable items. • Protection from loss of No Claim Bonus/No Claim Bonus Super if the total claim amount is less than 25% of SI. • Increase in SI every year based on inflation CPI index. Annual Health Check-up To regularly monitor the health and well being, of ea

In [48]:
question = "What are the benefits provided under the Care Plus health insurance plan?"

query_embedding = embedding_model.encode(question).tolist()

results = collection.query(
    query_embeddings=[query_embedding],
    n_results=5,
    where={"document_type": "Brochure"}
)

for i, document in enumerate(results["documents"][0], start=1):
    metadata = results["metadatas"][0][i - 1]
    print(f"\n--- Retrieved Chunk {i} ---")
    print("Source:", metadata["source"])
    print("Page:", metadata["page"])
    print("Document Type:", metadata["document_type"])
    print("Text:", document)


--- Retrieved Chunk 1 ---
Source: care-freedom-(health-insurance-product)---brochure.pdf
Page: 4
Document Type: Brochure
Text: more! Avail a discount on your premium if you opt for a 2-year policy term or a 3-year policy term. TAX BENEFIT Opting for health insurance is certainly a step in the right direction, and it comes with a two-fold beneﬁt. Not only does it ensure that you and your family can access good medical care at all times, it also enables you to avail a tax beneﬁt on the premiums you pay towards your health insurance, as per prevailing tax laws of the Income Tax Act, 1961. (Tax Beneﬁts under the policy will be as per

--- Retrieved Chunk 2 ---
Source: care-freedom-(health-insurance-product)---brochure.pdf
Page: 2
Document Type: Brochure
Text: Presenting the freedom to get Health Insurance Coverage without worrying about your current health status. This plan of Care Freedom oﬀers you the liberty to lead the life of your choice and stay worry-free, always. Under this plan, 

In [49]:
question = "What are the features of the Care Freedom health insurance product?"

query_embedding = embedding_model.encode(question).tolist()

results = collection.query(
    query_embeddings=[query_embedding],
    n_results=5,
    where={"document_type": "Brochure"}
)

for i, document in enumerate(results["documents"][0], start=1):
    metadata = results["metadatas"][0][i - 1]
    print(f"\n--- Retrieved Chunk {i} ---")
    print("Source:", metadata["source"])
    print("Page:", metadata["page"])
    print("Document Type:", metadata["document_type"])
    print("Text:", document)


--- Retrieved Chunk 1 ---
Source: care-freedom-(health-insurance-product)---brochure.pdf
Page: 2
Document Type: Brochure
Text: Presenting the freedom to get Health Insurance Coverage without worrying about your current health status. This plan of Care Freedom oﬀers you the liberty to lead the life of your choice and stay worry-free, always. Under this plan, you get • Only 2 year wait period in case of Pre-existing Diseases • Annual health check-up for all insured members covered as adults • Automatic recharge of base sum insured if claim amount exhausts your coverage limit • Ease of cashless treatment & settlement of claims

--- Retrieved Chunk 2 ---
Source: care-freedom-(health-insurance-product)---brochure.pdf
Page: 2
Document Type: Brochure
Text: if claim amount exhausts your coverage limit • Ease of cashless treatment & settlement of claims directly by the company • Life-long Renewability • Tax beneﬁt on premium paid, u/s 80 D of Income Tax Act – 1961* • Freedom to get health insu

In [50]:
question = "What is the claim process for the health insurance policy?"

query_embedding = embedding_model.encode(question).tolist()

results = collection.query(
    query_embeddings=[query_embedding],
    n_results=5,
    where={"document_type": "Policy Usage Guide"}
)

for i, document in enumerate(results["documents"][0], start=1):
    metadata = results["metadatas"][0][i - 1]
    print(f"\n--- Retrieved Chunk {i} ---")
    print("Source:", metadata["source"])
    print("Page:", metadata["page"])
    print("Document Type:", metadata["document_type"])
    print("Text:", document)


--- Retrieved Chunk 1 ---
Source: care-supreme----policy-usage-guide.pdf
Page: 1
Document Type: Policy Usage Guide
Text: Policy Usage Guide Note: This is a illustrative summary description of the health insurance policy cover for quick customer overview and does not in any way claim to present exhaustive information.Please refer to policy document for complete details. In-patient Care If the Insured Person is hospitalized for a minimum period of 24 consecutive hours, the Company pays for the expenses pertaining to in-patient treatment such as room charges, nursing expenses, intensive care unit charges, surgeon's fee, doctor's

--- Retrieved Chunk 2 ---
Source: care--policy-usage-guide.pdf
Page: 2
Document Type: Policy Usage Guide
Text: for 68 Non-Payable items. • Protection from loss of No Claim Bonus/No Claim Bonus Super if the total claim amount is less than 25% of SI. • Increase in SI every year based on inflation CPI index. Annual Health Check-up To regularly monitor the health an

In [51]:
import inspect

print(inspect.getsource(ask_rag))

def ask_rag(query):
    # Query ko embedding mein convert karna
    query_embedding = embedding_model.encode(query).tolist()

    # ChromaDB se top 5 relevant chunks retrieve karna
    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=5
    )

    retrieved_docs = results["documents"][0]
    context = "\n\n".join(retrieved_docs)

    # RAG prompt
    prompt = f"""
You are an insurance information assistant.

Answer the question using only the information provided in the context.

If the answer is not available in the context, say:
"I could not find the answer in the provided documents."

Context:
{context}

Question:
{query}

Answer:
"""

    # LLM se answer generate karna
    response = llm.invoke(prompt)

    return response.content



In [52]:
generated_answers = []

for item in evaluation_questions:
    question = item["question"]

    answer = ask_rag(question)

    # Contexts ko alag se retrieve karna
    query_embedding = embedding_model.encode(question).tolist()
    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=5
    )

    contexts = results["documents"][0]

    generated_answers.append({
        "question": question,
        "answer": answer,
        "contexts": contexts
    })

    print("\n" + "=" * 80)
    print("QUESTION:", question)
    print("GENERATED ANSWER:", answer)
    print("RETRIEVED CONTEXTS:")
    for i, context in enumerate(contexts, start=1):
        print(f"\nContext {i}:", context)


QUESTION: What is the waiting period for pre-existing diseases?
GENERATED ANSWER: The waiting period for pre-existing diseases is 24 months of continuous insurance coverage.
RETRIEVED CONTEXTS:

Context 1: Illnesses, conditions or Pre-Existing Diseases at the time of commencement of first policy with Us, any Claim in respect of that Illness, condition or Pre-existing Disease shall not be covered until the completion of 24 months of continuous insurance coverage with Us from the first Policy Period Start Date. 3. Pre-existing Disease: (Code - Excl 01) a. Expenses related to the treatment of a pre-existing Disease (PED) and its direct complications shall be excluded until the expiry of 24 months of continuous

Context 2: Insurance) Regulations, then waiting period for the same would be reduced to the extent of prior coverage. d. Coverage under the policy after the expiry of 24 months for any pre-existing disease is subject to the same being declared at the time of application and accept

In [53]:
import pandas as pd

evaluation_df = pd.DataFrame(generated_answers)

evaluation_df

,question,answer,contexts
0,What is the waiting period for pre-existing di...,The waiting period for pre-existing diseases i...,"[Illnesses, conditions or Pre-Existing Disease..."
1,What are the exclusions under the health insur...,"According to the provided context, the exclusi...",[12 CARE PLUS - UIN: CHIHLIP 22047 V 012122 4....
2,What are the benefits provided under the Care ...,I could not find the answer in the provided do...,[features (e.g. offering deductible and Co-pay...
3,What are the features of the Care Freedom heal...,The features of the Care Freedom health insura...,[Presenting the freedom to get Health Insuranc...
4,What is the claim process for the health insur...,The claim process for the health insurance pol...,[from the date of occurrence of event either a...


In [54]:
reference_answers = [
    "According to the Care Supreme Policy Usage Guide, the waiting period for pre-existing diseases (PED) is 36 months. The optional PED Waiting Period Modification benefit can modify this applicable waiting period.",
    "",
    "",
    "",
    ""
]

evaluation_df["reference"] = reference_answers

evaluation_df[["question", "answer", "reference"]]

,question,answer,reference
0,What is the waiting period for pre-existing di...,The waiting period for pre-existing diseases i...,According to the Care Supreme Policy Usage Gui...
1,What are the exclusions under the health insur...,"According to the provided context, the exclusi...",
2,What are the benefits provided under the Care ...,I could not find the answer in the provided do...,
3,What are the features of the Care Freedom heal...,The features of the Care Freedom health insura...,
4,What is the claim process for the health insur...,The claim process for the health insurance pol...,


In [55]:
%pip install ragas==0.2.15

In [56]:
%pip show ragas datasets pyarrow

Name: ragasNote: you may need to restart the kernel to use updated packages.

Version: 0.2.15
Summary: 
Home-page: 
Author: 
Author-email: 
License: 
Location: C:\Users\LENOVO 4HIN\anaconda3\Lib\site-packages
Requires: appdirs, datasets, diskcache, langchain, langchain-community, langchain-core, langchain_openai, nest-asyncio, numpy, openai, pydantic, tiktoken
Required-by: 
---
Name: datasets
Version: 5.0.1
Summary: HuggingFace community-driven open-source library of datasets
Home-page: https://github.com/huggingface/datasets
Author: HuggingFace Inc.
Author-email: thomas@huggingface.co
License: Apache 2.0
Location: C:\Users\LENOVO 4HIN\anaconda3\Lib\site-packages
Requires: dill, filelock, fsspec, httpx, huggingface-hub, multiprocess, numpy, packaging, pandas, pyarrow, pyyaml, requests, tqdm, xxhash
Required-by: ragas
---
Name: pyarrow
Version: 23.0.1
Summary: Python library for Apache Arrow
Home-page: https://arrow.apache.org/
Author: 
Author-email: 
License-Expression: Apache-2.0
Loca

In [57]:
import pyarrow
print("PyArrow version:", pyarrow.__version__)

import datasets
print("Datasets version:", datasets.__version__)

import ragas
print("RAGAS version:", ragas.__version__)

PyArrow version: 23.0.1
Datasets version: 5.0.1
RAGAS version: 0.2.15


In [58]:
import ragas
print("RAGAS version:", ragas.__version__)

RAGAS version: 0.2.15


In [59]:
from langchain_ollama import ChatOllama
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_community.embeddings import HuggingFaceEmbeddings

print("Evaluator imports successful!")

Evaluator imports successful!


In [60]:
# LLM evaluator
evaluator_llm = LangchainLLMWrapper(
    ChatOllama(model="llama3.2", temperature=0)
)

# Embedding model for evaluation
evaluator_embeddings = LangchainEmbeddingsWrapper(
    HuggingFaceEmbeddings(
        model_name="sentence-transformers/all-MiniLM-L6-v2"
    )
)

print("RAGAS evaluator configured successfully!")

C:\Users\LENOVO 4HIN\AppData\Local\Temp\ipykernel_25132\3539988020.py:8: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  HuggingFaceEmbeddings(


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

RAGAS evaluator configured successfully!


In [61]:
from datasets import Dataset

q1_data = {
    "question": [evaluation_questions[0]["question"]],
    "answer": [generated_answers[0]["answer"]],
    "contexts": [generated_answers[0]["contexts"]],
    "reference": [reference_answers[0]]
}

ragas_dataset = Dataset.from_dict(q1_data)

ragas_dataset

Dataset({
    features: ['question', 'answer', 'contexts', 'reference'],
    num_rows: 1
})

In [62]:
from ragas.metrics import (
    faithfulness,
    answer_relevancy,
    context_precision,
    context_recall
)

print("RAGAS metrics imported successfully!")

RAGAS metrics imported successfully!


In [63]:
# Evaluator LLM aur embeddings ko metrics ke saath configure karna
faithfulness.llm = evaluator_llm
answer_relevancy.llm = evaluator_llm
answer_relevancy.embeddings = evaluator_embeddings
context_precision.llm = evaluator_llm
context_recall.llm = evaluator_llm

print("RAGAS metrics configured successfully!")

RAGAS metrics configured successfully!


In [64]:

query = "List all exclusions under the Care Plus health insurance policy, including waiting periods, diseases, treatments, and procedures."

query_embedding = embedding_model.encode(query).tolist()

results = collection.query(
    query_embeddings=[query_embedding],
    n_results=15,
    where={
        "source": "care-plus--(health-insurance-product)---prospectus-cum-sales-literature.pdf"
    }
)

for i, (doc, metadata) in enumerate(
    zip(results["documents"][0], results["metadatas"][0]), start=1
):
    print(f"\n--- Chunk {i} ---")
    print("Source:", metadata.get("source"))
    print("Page:", metadata.get("page"))
    print(doc)


--- Chunk 1 ---
Source: care-plus--(health-insurance-product)---prospectus-cum-sales-literature.pdf
Page: 12
a specific exclusion. e. If the Insured Person is continuously covered without any break as defined under the applicable norms on portability stipulated by IRDAI, then waiting period for the same would be reduced to the extent of prior coverage. f. List of specific diseases/procedures: 1. Any treatment related to Arthritis (if non-infective), Osteoarthritis and Osteoporosis, Gout, Rheumatism, Spinal Disorders(unless caused by accident), Joint Replacement Surgery(unless caused by accident), Arthroscopic

--- Chunk 2 ---
Source: care-plus--(health-insurance-product)---prospectus-cum-sales-literature.pdf
Page: 12
12 CARE PLUS - UIN: CHIHLIP 22047 V 012122 4. EXCLUSIONS 4.1 Waiting Periods: (i) Pre-Existing Diseases: Code- Excl 01 a. Expenses related to the treatment of a pre-existing Disease (PED) and its direct complications shall be excluded until the expiry of 36 months of cont

In [65]:

query = "What are the benefits provided under the Care Plus health insurance plan?"

query_embedding = embedding_model.encode(query).tolist()

results = collection.query(
    query_embeddings=[query_embedding],
    n_results=10,
    where={
        "document_type": "Brochure"
    }
)

for i, (doc, metadata) in enumerate(
    zip(results["documents"][0], results["metadatas"][0]), start=1
):
    print(f"\n--- Chunk {i} ---")
    print("Source:", metadata.get("source"))
    print("Page:", metadata.get("page"))
    print(doc)


--- Chunk 1 ---
Source: care-freedom-(health-insurance-product)---brochure.pdf
Page: 4
more! Avail a discount on your premium if you opt for a 2-year policy term or a 3-year policy term. TAX BENEFIT Opting for health insurance is certainly a step in the right direction, and it comes with a two-fold beneﬁt. Not only does it ensure that you and your family can access good medical care at all times, it also enables you to avail a tax beneﬁt on the premiums you pay towards your health insurance, as per prevailing tax laws of the Income Tax Act, 1961. (Tax Beneﬁts under the policy will be as per

--- Chunk 2 ---
Source: care-freedom-(health-insurance-product)---brochure.pdf
Page: 2
Presenting the freedom to get Health Insurance Coverage without worrying about your current health status. This plan of Care Freedom oﬀers you the liberty to lead the life of your choice and stay worry-free, always. Under this plan, you get • Only 2 year wait period in case of Pre-existing Diseases • Annual heal

In [66]:

query = "What are the features of the Care Freedom health insurance product?"

query_embedding = embedding_model.encode(query).tolist()

results = collection.query(
    query_embeddings=[query_embedding],
    n_results=10,
    where={"document_type": "Brochure"}
)

for i, (doc, metadata) in enumerate(
    zip(results["documents"][0], results["metadatas"][0]), start=1
):
    print(f"\n--- Chunk {i} ---")
    print("Source:", metadata.get("source"))
    print("Page:", metadata.get("page"))
    print(doc)


--- Chunk 1 ---
Source: care-freedom-(health-insurance-product)---brochure.pdf
Page: 2
Presenting the freedom to get Health Insurance Coverage without worrying about your current health status. This plan of Care Freedom oﬀers you the liberty to lead the life of your choice and stay worry-free, always. Under this plan, you get • Only 2 year wait period in case of Pre-existing Diseases • Annual health check-up for all insured members covered as adults • Automatic recharge of base sum insured if claim amount exhausts your coverage limit • Ease of cashless treatment & settlement of claims

--- Chunk 2 ---
Source: care-freedom-(health-insurance-product)---brochure.pdf
Page: 2
if claim amount exhausts your coverage limit • Ease of cashless treatment & settlement of claims directly by the company • Life-long Renewability • Tax beneﬁt on premium paid, u/s 80 D of Income Tax Act – 1961* • Freedom to get health insurance cover without any hassles • Freedom of quality treatment at leading networ

In [67]:

query = "What is the claim process for the health insurance policy?"

query_embedding = embedding_model.encode(query).tolist()

results = collection.query(
    query_embeddings=[query_embedding],
    n_results=10,
    where={"document_type": "Policy Usage Guide"}
)

for i, (doc, metadata) in enumerate(
    zip(results["documents"][0], results["metadatas"][0]), start=1
):
    print(f"\n--- Chunk {i} ---")
    print("Source:", metadata.get("source"))
    print("Page:", metadata.get("page"))
    print(doc)


--- Chunk 1 ---
Source: care-supreme----policy-usage-guide.pdf
Page: 1
Policy Usage Guide Note: This is a illustrative summary description of the health insurance policy cover for quick customer overview and does not in any way claim to present exhaustive information.Please refer to policy document for complete details. In-patient Care If the Insured Person is hospitalized for a minimum period of 24 consecutive hours, the Company pays for the expenses pertaining to in-patient treatment such as room charges, nursing expenses, intensive care unit charges, surgeon's fee, doctor's

--- Chunk 2 ---
Source: care--policy-usage-guide.pdf
Page: 2
for 68 Non-Payable items. • Protection from loss of No Claim Bonus/No Claim Bonus Super if the total claim amount is less than 25% of SI. • Increase in SI every year based on inflation CPI index. Annual Health Check-up To regularly monitor the health and well being, of each insured member, they get one health check-up in every policy year. We have alr

In [68]:

evaluation_df[["question", "answer", "contexts", "reference"]]

,question,answer,contexts,reference
0,What is the waiting period for pre-existing di...,The waiting period for pre-existing diseases i...,"[Illnesses, conditions or Pre-Existing Disease...",According to the Care Supreme Policy Usage Gui...
1,What are the exclusions under the health insur...,"According to the provided context, the exclusi...",[12 CARE PLUS - UIN: CHIHLIP 22047 V 012122 4....,
2,What are the benefits provided under the Care ...,I could not find the answer in the provided do...,[features (e.g. offering deductible and Co-pay...,
3,What are the features of the Care Freedom heal...,The features of the Care Freedom health insura...,[Presenting the freedom to get Health Insuranc...,
4,What is the claim process for the health insur...,The claim process for the health insurance pol...,[from the date of occurrence of event either a...,


In [69]:

from ragas import evaluate

result = evaluate(
    dataset=ragas_dataset,
    metrics=[
        faithfulness,
        answer_relevancy,
        context_precision,
        context_recall
    ],
    llm=evaluator_llm,
    embeddings=evaluator_embeddings
)

result

Evaluating:   0%|          | 0/4 [00:00<?, ?it/s]

Exception raised in Job[0]: TimeoutError()
Exception raised in Job[2]: TimeoutError()
Exception raised in Job[1]: TimeoutError()


{'faithfulness': nan, 'answer_relevancy': nan, 'context_precision': nan, 'context_recall': 0.4000}

In [70]:

print("Question:", ragas_dataset[0]["question"])
print("\nAnswer:", ragas_dataset[0]["answer"])
print("\nReference:", ragas_dataset[0]["reference"])
print("\nNumber of contexts:", len(ragas_dataset[0]["contexts"]))
print("\nContexts:")
for i, context in enumerate(ragas_dataset[0]["contexts"], start=1):
    print(f"\nContext {i}:\n{context[:1000]}")

Question: What is the waiting period for pre-existing diseases?

Answer: The waiting period for pre-existing diseases is 24 months of continuous insurance coverage.

Reference: According to the Care Supreme Policy Usage Guide, the waiting period for pre-existing diseases (PED) is 36 months. The optional PED Waiting Period Modification benefit can modify this applicable waiting period.

Number of contexts: 5

Contexts:

Context 1:
Illnesses, conditions or Pre-Existing Diseases at the time of commencement of first policy with Us, any Claim in respect of that Illness, condition or Pre-existing Disease shall not be covered until the completion of 24 months of continuous insurance coverage with Us from the first Policy Period Start Date. 3. Pre-existing Disease: (Code - Excl 01) a. Expenses related to the treatment of a pre-existing Disease (PED) and its direct complications shall be excluded until the expiry of 24 months of continuous

Context 2:
Insurance) Regulations, then waiting period

In [71]:
result.to_pandas()

,user_input,retrieved_contexts,response,reference,faithfulness,answer_relevancy,context_precision,context_recall
0,What is the waiting period for pre-existing di...,"[Illnesses, conditions or Pre-Existing Disease...",The waiting period for pre-existing diseases i...,According to the Care Supreme Policy Usage Gui...,NaN,NaN,NaN,0.4


In [72]:
print("Generated answer:")
print(ragas_dataset[0]["answer"])

print("\nReference answer:")
print(ragas_dataset[0]["reference"])

print("\nContexts:")
for i, context in enumerate(ragas_dataset[0]["contexts"], start=1):
    print(f"\nContext {i}:")
    print(context)

Generated answer:
The waiting period for pre-existing diseases is 24 months of continuous insurance coverage.

Reference answer:
According to the Care Supreme Policy Usage Guide, the waiting period for pre-existing diseases (PED) is 36 months. The optional PED Waiting Period Modification benefit can modify this applicable waiting period.

Contexts:

Context 1:
Illnesses, conditions or Pre-Existing Diseases at the time of commencement of first policy with Us, any Claim in respect of that Illness, condition or Pre-existing Disease shall not be covered until the completion of 24 months of continuous insurance coverage with Us from the first Policy Period Start Date. 3. Pre-existing Disease: (Code - Excl 01) a. Expenses related to the treatment of a pre-existing Disease (PED) and its direct complications shall be excluded until the expiry of 24 months of continuous

Context 2:
Insurance) Regulations, then waiting period for the same would be reduced to the extent of prior coverage. d. Cove

In [73]:

query = "What is the waiting period for pre-existing diseases?"

query_embedding = embedding_model.encode(query).tolist()

results = collection.query(
    query_embeddings=[query_embedding],
    n_results=5,
    where={
        "source": "care-supreme----policy-usage-guide.pdf"
    }
)

for i, (doc, metadata) in enumerate(
    zip(results["documents"][0], results["metadatas"][0]), start=1
):
    print(f"\n--- Context {i} ---")
    print("Source:", metadata.get("source"))
    print("Page:", metadata.get("page"))
    print(doc)


--- Context 1 ---
Source: care-supreme----policy-usage-guide.pdf
Page: 3
provided is representative and summary of waiting period applicable under the policy, for complete details please refer to T&C WAITING PERIOD The time span during which you can not claim some or all benefits, specified for the illnesses covered under the insurance policy. A 30 day initial wait period applies at start of policy except for policy renewal & accident cases. For specified illness, treatment & surgeries for example – Hernia, Ulcer, ENT related, Arthirities, Ligament Tear, Cataract & Stone the

--- Context 2 ---
Source: care-supreme----policy-usage-guide.pdf
Page: 3
for example – Hernia, Ulcer, ENT related, Arthirities, Ligament Tear, Cataract & Stone the applicant will wait 24 month and for pre-existing aliment 36 months. Clause 4.1(a) (i), (ii), (iii)

--- Context 3 ---
Source: care-supreme----policy-usage-guide.pdf
Page: 3
PED waiting period related to Diabetes/ Hypertension/ Hyperlipidemia/ Asthma, 

In [74]:

# Generate Q1's answer using only the Care Supreme Policy Usage Guide
q1_query = evaluation_questions[0]["question"]
q1_embedding = embedding_model.encode(q1_query).tolist()

q1_results = collection.query(
    query_embeddings=[q1_embedding],
    n_results=5,
    where={"source": "care-supreme----policy-usage-guide.pdf"}
)

q1_contexts = q1_results["documents"][0]
q1_context_text = "\n\n".join(q1_contexts)

q1_prompt = f"""
You are an insurance information assistant.
Answer using only the provided context.
If the answer is not available, say:
"I could not find the answer in the provided documents."

Context:
{q1_context_text}

Question:
{q1_query}

Answer:
"""

q1_response = llm.invoke(q1_prompt)
q1_answer = q1_response.content

print("Question:", q1_query)
print("\nGenerated answer:", q1_answer)
print("\nRetrieved contexts:")
for i, context in enumerate(q1_contexts, start=1):
    print(f"\nContext {i}:\n{context}")

Question: What is the waiting period for pre-existing diseases?

Generated answer: The waiting period for pre-existing diseases is 36 months.

Retrieved contexts:

Context 1:
provided is representative and summary of waiting period applicable under the policy, for complete details please refer to T&C WAITING PERIOD The time span during which you can not claim some or all benefits, specified for the illnesses covered under the insurance policy. A 30 day initial wait period applies at start of policy except for policy renewal & accident cases. For specified illness, treatment & surgeries for example – Hernia, Ulcer, ENT related, Arthirities, Ligament Tear, Cataract & Stone the

Context 2:
for example – Hernia, Ulcer, ENT related, Arthirities, Ligament Tear, Cataract & Stone the applicant will wait 24 month and for pre-existing aliment 36 months. Clause 4.1(a) (i), (ii), (iii)

Context 3:
PED waiting period related to Diabetes/ Hypertension/ Hyperlipidemia/ Asthma, at the time of issuance

In [75]:

# Update Q1 values using .at to assign the whole list to one cell
evaluation_df.at[0, "answer"] = q1_answer
evaluation_df.at[0, "contexts"] = q1_contexts

# Rebuild the Q1 RAGAS dataset
from datasets import Dataset

q1_data = {
    "question": [evaluation_df.at[0, "question"]],
    "answer": [evaluation_df.at[0, "answer"]],
    "contexts": [evaluation_df.at[0, "contexts"]],
    "reference": [evaluation_df.at[0, "reference"]]
}

ragas_dataset = Dataset.from_dict(q1_data)

print("Updated Q1 evaluation dataset:")
print(ragas_dataset[0])


Updated Q1 evaluation dataset:
{'question': 'What is the waiting period for pre-existing diseases?', 'answer': 'The waiting period for pre-existing diseases is 36 months.', 'contexts': ['provided is representative and summary of waiting period applicable under the policy, for complete details please refer to T&C WAITING PERIOD The time span during which you can not claim some or all benefits, specified for the illnesses covered under the insurance policy. A 30 day initial wait period applies at start of policy except for policy renewal & accident cases. For specified illness, treatment & surgeries for example – Hernia, Ulcer, ENT related, Arthirities, Ligament Tear, Cataract & Stone the', 'for example – Hernia, Ulcer, ENT related, Arthirities, Ligament Tear, Cataract & Stone the applicant will wait 24 month and for pre-existing aliment 36 months. Clause 4.1(a) (i), (ii), (iii)', "PED waiting period related to Diabetes/ Hypertension/ Hyperlipidemia/ Asthma, at the time of issuance of fi

In [76]:

from ragas import evaluate

result_q1 = evaluate(
    dataset=ragas_dataset,
    metrics=[
        faithfulness,
        answer_relevancy,
        context_precision,
        context_recall
    ],
    llm=evaluator_llm,
    embeddings=evaluator_embeddings
)

print(result_q1)

Evaluating:   0%|          | 0/4 [00:00<?, ?it/s]

Exception raised in Job[2]: TimeoutError()
Exception raised in Job[1]: TimeoutError()


{'faithfulness': 1.0000, 'answer_relevancy': nan, 'context_precision': nan, 'context_recall': 0.6667}


In [77]:

q1_scores = result_q1.to_pandas()

q1_scores

,user_input,retrieved_contexts,response,reference,faithfulness,answer_relevancy,context_precision,context_recall
0,What is the waiting period for pre-existing di...,[provided is representative and summary of wai...,The waiting period for pre-existing diseases i...,According to the Care Supreme Policy Usage Gui...,1.0,NaN,NaN,0.666667


In [78]:

reference_answers[1] = """According to the Care Plus policy document, expenses related to pre-existing diseases and their direct complications are excluded until 36 months of continuous coverage have expired from the inception of the first policy. If the sum insured is enhanced, the exclusion applies afresh to the increased amount. Continuous prior coverage may reduce the waiting period according to portability norms. The retrieved passages also list treatment related to non-infective arthritis, osteoarthritis, osteoporosis, gout, rheumatism, spinal disorders (unless caused by an accident), joint replacement surgery (unless caused by an accident), and arthroscopic procedures. These are the exclusions supported by the retrieved passages; they do not establish the complete list of policy exclusions."""

evaluation_df.at[1, "reference"] = reference_answers[1]

print(evaluation_df.loc[1, ["question", "reference"]])

question     What are the exclusions under the health insur...
reference    According to the Care Plus policy document, ex...
Name: 1, dtype: object


In [79]:
# Retrieve Q2 context only from the Care Plus prospectus
q2_query = evaluation_questions[1]["question"]
q2_query_embedding = embedding_model.encode(q2_query).tolist()

q2_results = collection.query(
    query_embeddings=[q2_query_embedding],
    n_results=5,
    where={
        "source": "care-plus--(health-insurance-product)---prospectus-cum-sales-literature.pdf"
    }
)

q2_contexts = q2_results["documents"][0]
q2_context = "\n\n".join(q2_contexts)

q2_prompt = f"""
You are an insurance information assistant.
Answer the question using only the information provided in the context.
If the answer is not available in the context, say:
"I could not find the answer in the provided documents."

Context:
{q2_context}

Question:
{q2_query}

Answer:
"""

q2_response = llm.invoke(q2_prompt)
q2_answer = q2_response.content

# Update Q2 answer and contexts in the evaluation data
generated_answers[1]["answer"] = q2_answer
generated_answers[1]["contexts"] = q2_contexts

evaluation_df.at[1, "answer"] = q2_answer
evaluation_df.at[1, "contexts"] = q2_contexts

print("Q2 Answer:\n", q2_answer)
print("\nNumber of retrieved context chunks:", len(q2_contexts))

Q2 Answer:
 According to the provided context, the exclusions under the health insurance policy are:

1. Pre-Existing Diseases (Code- Excl 01):
	* Expenses related to the treatment of a pre-existing disease (PED) and its direct complications shall be excluded until the expiry of 36 months of continuous coverage after the date of inception of the first policy with insurer.
2. Specific diseases/procedures (Code- Excl 02):
	* Expenses related to the treatment of the listed Conditions, surgeries/treatments shall be excluded until the expiry of 24 months of continuous coverage, as may be the case after the date of inception of the first policy with the Company.
	* This exclusion shall not be applicable for claims arising due to an accident.

Note that there are also other exclusions mentioned in the context, such as the Claim being paid only for the Policy Year in which the Insured event occurs, and the Policy not covering losses other than the covered benefits. However, the above two exclu

In [2]:
print("Q2 Answer:\n", q2_answer)
print("\nRetrieved context chunks:", len(q2_contexts))

print("\nEvaluation table:")
display(evaluation_df.loc[1:1, ["question", "answer", "reference"]])

NameError: name 'q2_answer' is not defined